# Agent的高级用法-ToolStrategy
## 1、ToolStrategy的多种结构化输出方式：schema参数

### 1.1 Pydantic类型

使用CloseAI平台的模型

In [4]:
from dataclasses import dataclass

from certifi import contents
from defusedxml.cElementTree import parse
from fastmcp.utilities.docstring_parsing import parse_docstring
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langchain_core.messages import SystemMessage
from langchain_core.tools import tool

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking":{"type":"disabled"}}
)

使用OpenRounter平台的模型

In [2]:
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("DEEPSEEK_API_KEY")
OPENROUTER_API_BASE = os.getenv("DEEPSEEK_BASE_URL")

model = ChatOpenRouter(
    model="deepseek/deepseek-v4-flash",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

举例1：

In [ ]:
from pydantic import BaseModel,Field
from rich import print as rprint


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户姓名")
    email : str = Field(description="用户邮箱地址")
    phone : str = Field(description="用户的手机号")

agent = create_agent(
    model=model,
    response_format = ToolStrategy(schema=ContactInfo),
)

response = agent.invoke({
    "messages":[
        HumanMessage(content="从这段话中抽取结构信息：小明的邮箱地址是shkstart@atguigu.com，电话：13191391293")
    ]
})

rprint(response)

举例2：添加工具的调用

In [ ]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str):客户查询字符串，例如“张三”或者“李四”

    Returns:
        str:客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-10，累计消费：$3,200"
    else:
        return  f"关于客户{query}无记录"

@tool(parse_docstring=True)
def send_email(customer : str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向{customer}发送感谢邮件"

# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name : str = Field(None,description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

agent = create_agent(
    model = model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis),
)

result = agent.invoke({
    "messages":[{"role":"user","content":"请分析客户张三"}]
})

rprint(result)

### 1.2 TypedDict类型

举例1：

In [13]:
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(TypedDict):
    """用户的联系方式"""
    name: Annotated[str, ..., "用户姓名"]   ## ...表示必须填写的意思
    email: Annotated[str, ..., "用户邮箱地址"]
    phone: Annotated[str, ..., "用户的手机号"]

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo)
)

response = agent.invoke({
    "messages" : [
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234")
    ]
})


rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱是shkstart@atguigu.com,电话是：13012341234',
            additional_kwargs={},
            response_metadata={},
            id='e8d65a62-a017-4089-8ab1-aaaf3316aefd'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 327,
                    'total_tokens': 405,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 327
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '675424a9-5e75-45aa-9aac-1fbdf6981cc8',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a022dd-be10-70f3-9e1d-6f855e4883f1-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_00_vzZepQdXIrVZQhZoJhRs6021',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 327,
                'output_tokens': 78,
                'total_tokens': 405,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': 
'13012341234'}",
            name='ContactInfo',
            id='952b360a-687f-4d51-923e-a6589682eeaa',
            tool_call_id='call_00_vzZepQdXIrVZQhZoJhRs6021'
        )
    ],
    'structured_response': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'}
}

举例2：

In [ ]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 使用 TypedDict 定义客户分析报告 Schema
class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name: Annotated[Optional[str], None, "客户姓名"]
    customer_tier: Annotated[Literal["潜在客户", "普通客户", "VIP客户", "流失风险"], "潜在客户", "客户等级"]
    recent_activity: Annotated[Optional[str], None, "最近活动"]
    spending_level: Annotated[Optional[Literal["低", "中", "高"]], None, "消费水平"]
    send_email: Annotated[bool, False, "是否已发送感谢邮件"]

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                        "请分析指定客户的情况："
                                        "1. 先搜索客户数据库了解最新情况 "
                                        "2. 如果是VIP客户，则发送感谢邮件 "
                                        "3. 基于搜索结果生成结构化分析报告 "
                                        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)

# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

### 1.5 多schema联合模式

举例

In [15]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=Union[ContactInfo, EventInfo])
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
        ]
    }
)

for msg in response["messages"]:
    msg.pretty_print()

# print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_PwDG4teSKzAPSjzNQ9zc7217)
 Call ID: call_00_PwDG4teSKzAPSjzNQ9zc7217
  Args:
    name: 小明
    email: shkstart@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'


## 2、自定义工具消息：tool_message_content参数

举例：

In [16]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='caeed1a9-7643-432e-8b71-e918c18503bf'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 77,
                    'prompt_tokens': 351,
                    'total_tokens': 428,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 95
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'aafaae3b-c57d-4cf6-aca3-b4097a61c5c5',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a022f7-b048-7840-a570-737fb73ce844-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_NCg7OIntW28st28ii2Vr9600',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 351,
                'output_tokens': 77,
                'total_tokens': 428,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='d0a1bd8f-cfc0-4551-ba83-99e62a8cf2e3',
            tool_call_id='call_00_NCg7OIntW28st28ii2Vr9600'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

作为对比：

In [17]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema = ContactInfo,
        tool_message_content="已成功抽取信息"
    )
)

response = agent.invoke({
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='866e5d29-0e79-41b7-926b-6f0633c4bc3c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 77,
                    'prompt_tokens': 351,
                    'total_tokens': 428,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 95
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'a2ccf24e-6b56-4ee4-b296-642d01990914',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a022f8-4329-79d3-b5ce-696980926221-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_c1YRij2XYxkvlfcpVP7R7085',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 351,
                'output_tokens': 77,
                'total_tokens': 428,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已成功抽取信息',
            name='ContactInfo',
            id='9c1b5100-3fe5-4ea3-a80f-c4b41e0a6de2',
            tool_call_id='call_00_c1YRij2XYxkvlfcpVP7R7085'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}